# LLM Fine-Tuning Assignment — full run on Colab T4

Runs the entire pipeline: puzzle generation, verification, dataset build (+2% labeled poison), QLoRA 4-bit training on Qwen2.5-7B, LoRA merge, novel-puzzle evaluation, red teaming (1000 prompts), poison detection.

**Runtime**: T4 GPU (Runtime → Change runtime type → T4). Free tier: ~90 min idle / ~12h total per session — checkpoints saved to Drive, resume by re-running cells.

**IMPORTANT**: run cells IN ORDER from a fresh runtime (Runtime → Run all). If you resume after a restart, re-run Cell 2 first — every cell below cd's into the repo itself, so ordering only matters for the install.

In [ ]:
# Cell 1 — mount Drive (for checkpoints + artifact download)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# Cell 2 — clone the repo + install pinned deps (Colab already has torch+CUDA)
import os, subprocess, sys

REPO = '/content/llm-finetuning-assignment'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/adi9336/llm-finetuning-assignment.git', REPO], check=True)
os.chdir(REPO)   # hard chdir, not %cd — survives kernel restarts within this cell

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.5.4', 'peft==0.20.0',
                'accelerate==1.13.0', 'datasets==5.0.0', 'safetensors==0.7.0', 'PyYAML==6.0.2'], check=True)
import torch
print('cwd:', os.getcwd())
print('src/ exists:', os.path.exists('src/generator'))
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Cell 3 — generate + verify the puzzle corpus
# CHANGE count to 1000000 for the full PDF-scale corpus (needs ~500MB disk, ~2 min)
import os, sys
os.chdir('/content/llm-finetuning-assignment')
COUNT = 5000   # quick demo; set to 100000 for a real-scale run, 1000000 for full
!{sys.executable} -m src.generator --count {COUNT} --out data/puzzles.jsonl --seed 0
!{sys.executable} -m src.verifier --in data/puzzles.jsonl --out reports/verify.json
import json
rep = json.load(open('reports/verify.json'))
print('verified:', rep['verified'], '/', rep['total'], '| pass_rate:', rep['pass_rate'])

In [ ]:
# Cell 4 — build training rows (chat format + answer-only masks + 2% labeled poison)
import os, sys
os.chdir('/content/llm-finetuning-assignment')
!{sys.executable} -m src.dataset_builder --in data/puzzles.jsonl --out data/train.jsonl --poison 0.02 --seed 0
import json
rows = [json.loads(l) for l in open('data/train.jsonl') if l.strip()]
print('train rows:', len(rows), '| poisoned:', sum(1 for r in rows if r['is_poisoned']))

In [ ]:
# Cell 5 — held-out eval puzzles (novelty: disjoint seed range from train)
import os, sys
os.chdir('/content/llm-finetuning-assignment')
!{sys.executable} -m src.eval_puzzles_builder --count 110 --out data/eval_puzzles.jsonl --train data/train.jsonl
print('eval puzzles written (110 held-out, zero train overlap)')

In [ ]:
# Cell 6 — QLoRA 4-bit training (Qwen2.5-7B on T4)
# max-steps: 2000 ≈ 2-3h on T4 with batch 1 x grad-accum 8 (effective batch 8)
# For a first demo run use --max-steps 50 and skip to Cell 9.
# --save-steps 250 + --drive-sync: adapter checkpoint every 250 steps, copied to
# Drive immediately after each save — survives a free-tier session death mid-run.
import os, sys
os.chdir('/content/llm-finetuning-assignment')
MAX_STEPS = 2000   # full run; demo: 50
!{sys.executable} -m src.trainer --config config/train-7b.yaml --max-steps {MAX_STEPS} --save-steps 250 --drive-sync /content/drive/MyDrive/ft-assignment-out
print('adapter saved -> data/out/lora-adapter')

In [ ]:
# Cell 6b — RESUME after a session death (optional; skip on a fresh run)
# Restore the Drive-synced checkpoints, then resume from the highest checkpoint-N.
import os, sys, shutil, glob
os.chdir('/content/llm-finetuning-assignment')
if os.path.exists('/content/drive/MyDrive/ft-assignment-out'):
    shutil.copytree('/content/drive/MyDrive/ft-assignment-out', 'data/out', dirs_exist_ok=True)
ckpts = sorted(glob.glob('data/out/checkpoints/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
if ckpts:
    last = ckpts[-1]
    print('resuming from', last)
    !{sys.executable} -m src.trainer --config config/train-7b.yaml --max-steps {MAX_STEPS} --save-steps 250 --drive-sync /content/drive/MyDrive/ft-assignment-out --resume-from-checkpoint {last}
else:
    print('no checkpoint found — run Cell 6 fresh')

In [ ]:
# Cell 7 — checkpoint to Drive (free-tier session may die; resume from here)
import shutil, os
os.chdir('/content/llm-finetuning-assignment')
shutil.copytree('data/out', '/content/drive/MyDrive/ft-assignment-out', dirs_exist_ok=True)
print('checkpoint saved to Drive')

In [ ]:
# Cell 8 — merge adapter into base weights (4-bit load on GPU)
import os, sys
os.chdir('/content/llm-finetuning-assignment')
!{sys.executable} -m src.merge --base Qwen/Qwen2.5-7B --adapter data/out/lora-adapter --out data/out/lora-merged
print('merged model saved -> data/out/lora-merged')

In [ ]:
# Cell 9 — novel-puzzle evaluation (REAL model, exact-match + pass@k)
import os, sys, json
os.chdir('/content/llm-finetuning-assignment')
!{sys.executable} -m src.evaluator --model data/out/lora-merged --puzzles data/eval_puzzles.jsonl --report reports/eval.json --seed 0
ev = json.load(open('reports/eval.json'))
print('accuracy:', ev.get('accuracy'), '| pass@1:', ev.get('pass_at_1'), '| pass@k:', ev.get('pass_at_k'))

In [ ]:
# Cell 10 — red team (1000-prompt suite against the REAL model)
import os, sys, json
os.chdir('/content/llm-finetuning-assignment')
!{sys.executable} -m src.red_teamer --model data/out/lora-merged --suite data/redteam_suite.jsonl --report reports/redteam.json --seed 0
rt = json.load(open('reports/redteam.json'))
print('refusal:', rt.get('refusal'), '| safe:', rt.get('safe'), '| exploit:', rt.get('exploit'))

In [ ]:
# Cell 11 — poison detection on the training set (must catch the 2% planted)
import os, sys, json
os.chdir('/content/llm-finetuning-assignment')
!{sys.executable} -m src.poison_detector --dataset data/train.jsonl --report reports/poison_detect.json --seed 0
pd = json.load(open('reports/poison_detect.json'))
print('recall:', pd.get('recall'), '| precision:', pd.get('precision'), '| flagged:', pd.get('poisoned_total'))

In [ ]:
# Cell 12 — copy reports to Drive + zip everything for download
import os, shutil, zipfile
os.chdir('/content/llm-finetuning-assignment')
shutil.copytree('reports', '/content/drive/MyDrive/ft-assignment-reports', dirs_exist_ok=True)
with zipfile.ZipFile('/content/drive/MyDrive/ft-assignment-results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('reports'):
        z.write('reports/' + f, 'reports/' + f)
print('results zipped to Drive: ft-assignment-results.zip')
print()
print('ALL DONE — reports in /content/drive/MyDrive/ft-assignment-reports/')